# CME Futures: Risk Overlays

For each return horizon, this notebook selects the highest validation Sharpe from the immutable
union of signal and allocation results, then applies every position-level risk rule declared in
the case-study configuration. Stop-loss, trailing-stop, and time-exit parameters are fixed before
the validation backtest. They are not calibrated from the same validation price path they assess.

Risk rules execute inside the existing futures engine after product-keyed target decisions cross
the typed boundary. Every declared rule must finish, and the resulting per-label candidate sets
remain eligible for final validation selection.

In [1]:
"""Run the declared CME futures risk-overlay population."""

from case_studies.cme_futures.research_workflow import (
    ALL_LABELS,
    create_label_candidate_sets,
    open_study,
    pre_overlay_results,
    product_universe_table,
    rank_by_validation_sharpe,
    run_official_backtest_requests,
    strategy_request_frame,
)
from case_studies.utils.sweep_config import get_position_risk_controls

In [2]:
EXECUTION_TIER = "canonical"
WORKSPACE: str | None = None
PREVIEW_LABELS: list[str] = []

## Fixed per-label inputs and risk rules

No candidate cap or runtime-dependent skip is allowed. The configured list is the population.

In [3]:
study = open_study(execution_tier=EXECUTION_TIER, workspace=WORKSPACE)
if EXECUTION_TIER == "canonical":
    if PREVIEW_LABELS:
        raise ValueError("canonical execution cannot declare preview reductions")
    labels = ALL_LABELS
elif EXECUTION_TIER == "preview":
    if WORKSPACE is None or not PREVIEW_LABELS:
        raise ValueError("preview execution requires WORKSPACE and PREVIEW_LABELS")
    unknown = sorted(set(PREVIEW_LABELS) - set(ALL_LABELS))
    if unknown:
        raise ValueError(f"preview labels this case study does not declare: {unknown}")
    labels = tuple(PREVIEW_LABELS)
else:
    raise ValueError(f"unsupported execution tier: {EXECUTION_TIER!r}")
universe = product_universe_table()
universe

sector,product,expiry_rule,contract_months
str,str,str,str
"""agriculture""","""ZC""","""business_day_before_15th""","""H,K,N,U,Z"""
"""agriculture""","""ZL""","""business_day_before_15th""","""F,H,K,N,Q,U,V,Z"""
"""agriculture""","""ZM""","""business_day_before_15th""","""F,H,K,N,Q,U,V,Z"""
"""agriculture""","""ZS""","""business_day_before_15th""","""F,H,K,N,Q,U,X"""
"""agriculture""","""ZW""","""business_day_before_15th""","""H,K,N,U,Z"""
…,…,…,…
"""metals""","""SI""","""3rd_last_business_day""","""H,K,N,U,Z"""
"""treasuries""","""ZB""","""last_business_day""","""H,M,U,Z"""
"""treasuries""","""ZF""","""last_business_day""","""H,M,U,Z"""


In [4]:
risk_controls = get_position_risk_controls("cme_futures")
if not risk_controls:
    raise ValueError("the configured position-risk population is empty")

request_rows = []
for label in labels:
    selected = rank_by_validation_sharpe(
        study, pre_overlay_results(study, label=label, execution_tier=EXECUTION_TIER)
    )[0]
    strategy = selected.spec()["strategy"]
    prediction_hash = selected.registry_record()["prediction_hash"]
    for control in risk_controls:
        rule = {key: value for key, value in control.items() if key != "name"}
        request_rows.append(
            {
                "request_name": f"{selected.hash}-risk-{control['name']}",
                "prediction_hash": prediction_hash,
                "label": label,
                "signal": strategy["signal"],
                "allocation": strategy.get("allocation"),
                "risk": {"position_rules": [rule]},
                "costs": None,
                "chapter": "ch19",
            }
        )
requests = strategy_request_frame(request_rows)
requests.select("request_name", "prediction_hash", "label", "risk")

request_name,prediction_hash,label,risk
str,str,str,object
"""da43932722bc-risk-stop_loss_3p…","""a3f6b9092f0f""","""fwd_ret_5d""","{'position_rules': [{'type': 'stop_loss', 'threshold': 0.03}]}"
"""da43932722bc-risk-stop_loss_5p…","""a3f6b9092f0f""","""fwd_ret_5d""","{'position_rules': [{'type': 'stop_loss', 'threshold': 0.05}]}"
"""da43932722bc-risk-stop_loss_10…","""a3f6b9092f0f""","""fwd_ret_5d""","{'position_rules': [{'type': 'stop_loss', 'threshold': 0.1}]}"
"""da43932722bc-risk-stop_loss_15…","""a3f6b9092f0f""","""fwd_ret_5d""","{'position_rules': [{'type': 'stop_loss', 'threshold': 0.15}]}"
"""da43932722bc-risk-trailing_1pc…","""a3f6b9092f0f""","""fwd_ret_5d""","{'position_rules': [{'type': 'trailing_stop', 'threshold': 0.01}]}"
…,…,…,…
"""61d5b0551729-risk-trailing_15p…","""fbeb779467d5""","""fwd_ret_21d""","{'position_rules': [{'type': 'trailing_stop', 'threshold': 0.15}]}"
"""61d5b0551729-risk-trailing_20p…","""fbeb779467d5""","""fwd_ret_21d""","{'position_rules': [{'type': 'trailing_stop', 'threshold': 0.2}]}"
"""61d5b0551729-risk-time_exit_10""","""fbeb779467d5""","""fwd_ret_21d""","{'position_rules': [{'type': 'time_exit', 'bars': 10}]}"


## Execute and freeze risk candidates

Each request carries the fitted prediction checkpoint, product decisions, fold-transition policy,
contract and roll inputs, and one risk rule. Missing members fail before the candidate set exists.

In [5]:
execution = run_official_backtest_requests(
    study,
    requests,
    population_name="cme_futures-risk-validation-v1" if EXECUTION_TIER == "canonical" else None,
)
candidate_sets = (
    create_label_candidate_sets(study, execution, stage="risk")
    if EXECUTION_TIER == "canonical"
    else {}
)

`source` says whether each member was computed by this run or served from the registry because
an identical identity was already recorded. A re-run of a registered sweep is entirely `reused`
and completes in seconds; without the column that is indistinguishable from having computed
every row.

In [6]:
execution.catalog_rows.sort("label", "request_name")

request_name,label,prediction_hash,decision_hash,backtest_hash,complete,source
str,str,str,str,str,bool,str
"""61d5b0551729-risk-stop_loss_10…","""fwd_ret_21d""","""fbeb779467d5""","""b04ccb8859f7""","""0530ba0ae64c""",true,"""computed"""
"""61d5b0551729-risk-stop_loss_15…","""fwd_ret_21d""","""fbeb779467d5""","""1e62bdac8761""","""721f80d0bd29""",true,"""computed"""
"""61d5b0551729-risk-stop_loss_3p…","""fwd_ret_21d""","""fbeb779467d5""","""76f97c6d8672""","""92b1c66efeee""",true,"""computed"""
"""61d5b0551729-risk-stop_loss_5p…","""fwd_ret_21d""","""fbeb779467d5""","""291ec795f90e""","""86baf9c78c3f""",true,"""computed"""
"""61d5b0551729-risk-time_exit_10""","""fwd_ret_21d""","""fbeb779467d5""","""5b35c6d72cb2""","""4132838ae50a""",true,"""computed"""
…,…,…,…,…,…,…
"""da43932722bc-risk-trailing_1pc…","""fwd_ret_5d""","""a3f6b9092f0f""","""6c3ed313305b""","""2703d9d8a208""",true,"""computed"""
"""da43932722bc-risk-trailing_20p…","""fwd_ret_5d""","""a3f6b9092f0f""","""890d29ef0739""","""bdc8ba13c71a""",true,"""computed"""
"""da43932722bc-risk-trailing_2pc…","""fwd_ret_5d""","""a3f6b9092f0f""","""215afc7a7b35""","""619a5cd773b4""",true,"""computed"""


Final selection in `19_strategy_analysis` uses the union of signal, allocation, and risk-overlay
results. Cost-sensitivity rows are excluded.